In [ ]:
!pip install -q transformers accelerate torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
# Choose the Qwen 2.5 1.5B Instruct model
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model with automatic dtype and device placement
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
model.eval()

print("Model loaded successfully.")
print("Device map:", model.hf_device_map if hasattr(model, "hf_device_map") else "Single device")

In [ ]:
def generate_reply(chat_history, user_message, max_new_tokens=256):
    # Append the new user message to the history
    chat_history.append({"role": "user", "content": user_message})
    # Apply Qwen chat template to convert messages into a single text prompt
    prompt_text = tokenizer.apply_chat_template(
        chat_history,
        tokenize=False,
        add_generation_prompt=True,  # model expects a generation prompt
    )
    # Tokenize the prompt
    inputs = tokenizer(
        [prompt_text],
        return_tensors="pt"
    ).to(model.device)
    # Generate the model output
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05
        )

    # Extract only the newly generated tokens
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    # Decode the assistant's reply
    assistant_reply = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()
    # Append assistant reply to history
    chat_history.append({"role": "assistant", "content": assistant_reply})
    return chat_history, assistant_reply

In [ ]:
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    # Initialize chat history with a system message to set behavior
    chat_history = [
        {
            "role": "system",
            "content": "You are a helpful, concise AI assistant. Answer clearly and politely."
        }
    ]
    while True:
        user_input = input("User: ").strip()

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day.")
            break

        # Skip empty input
        if not user_input:
            print("Chatbot: Please type something so I can respond.")
            continue
        # Generate reply from the model
        chat_history, assistant_reply = generate_reply(chat_history, user_input)
        # Basic safety: fallback message if reply is empty
        if assistant_reply == "":
            assistant_reply = "I'm not sure how to respond to that yet, but I'm learning."
        print("Chatbot:", assistant_reply)
run_chatbot()